##1. Importar librerias y configurar Spark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, struct, to_timestamp
import pyspark.sql.functions as F

# Inicializar SparkSession para el pipeline de análisis de tráfico marítimo y profundidades
spark = SparkSession.builder.appName("vessel-traffic-depth-monitoring").getOrCreate()

##2. Leer datos - Marine Cadastre (AIS)

In [0]:
# Leer archivos AIS (Marine Cadastre) desde el volumen de CloudLabs
# Usamos un solo archivo para exploración inicial
ais_path = "/Volumes/proyecto/default/raw_data/marine-cadastre/ais-2025-01-01.csv.zst"

# Leer el archivo comprimido
ais_df = spark.read.csv(ais_path, header=True, inferSchema=True)

# Mostrar primeras filas
ais_df.show(5)

+---------+-------------------+----------+--------+---+-----+-------+--------------+----------+---------+-----------+------+------+-----+-----+-----+-----------+
|     mmsi|     base_date_time| longitude|latitude|sog|  cog|heading|   vessel_name|       imo|call_sign|vessel_type|status|length|width|draft|cargo|transceiver|
+---------+-------------------+----------+--------+---+-----+-------+--------------+----------+---------+-----------+------+------+-----+-----+-----+-----------+
|671087100|2025-01-01 00:00:00| -66.10297|18.46281|0.0|176.7|   NULL|WATER SPIRIT 2|IMO9212424|    5VGA7|         70|     0|    70|   18|  4.0|   70|          A|
|367733950|2025-01-01 00:00:00|-122.60927|48.48503|0.0|215.5|    115|      INVICTUS|      NULL|  WDI7962|         37|  NULL|    10|    3| NULL| NULL|          B|
|368138010|2025-01-01 00:00:02| -73.84652|40.47715|5.5|286.9|    289|      NEW YORK|IMO9005839|  WDL5112|         50|     0|    58|   14| NULL|   50|          A|
|367637210|2025-01-01 00:00:

##3. Extraer datos - NOAA Bathymetry

**3.1. Limitación de Acceso a S3 y Solución Implementada**

**El Problema**

CloudLabs (cluster compartido) no tiene permisos de red para acceder directamente a buckets S3 externos, incluso si son públicos. Los intentos de lectura directa desde Spark resultaban en error: `[INSUFFICIENT_PERMISSIONS] User does not have permission SELECT on any file`.

**La Solución Planteada**

Descargar los datos NOAA localmente usando AWS CLI (con flag `--no-sign-request` para acceso público), luego subirlos a CloudLabs mediante el upload de Databricks. Esto convierte los datos en archivos locales dentro de CloudLabs, eliminando la necesidad de acceso directo a S3.

**Fase 0: Instalación de AWS CLI**

Se ejecutó `brew install awscli` en terminal para instalar AWS CLI, herramienta de línea de comandos que permite descargar archivos desde buckets S3 públicos sin credenciales.

Verificación: `aws --version` → aws-cli/2.31.37

**Fase 1: Descarga desde S3 (Terminal)**

Se ejecutó: `aws s3 cp s3://noaa-dcdb-bathymetry-pds/csb/csv/2025/01/01/ ~/noaa_data/ --recursive --no-sign-request`

Este comando descargó todos los archivos CSV de profundidades marinas (NOAA Crowdsourced Bathymetry) del 1 de enero de 2025 a la computadora local. El flag `--no-sign-request` permite acceso público sin autenticación. Resultado: 252 archivos CSV (~381 MB) en `~/noaa_data/`

**Fase 2: Crear Infraestructura en CloudLabs**

Paso 1 - Schema: Se creó un schema llamado `vessel_traffic_monitoring` en el catalog `labs_56754_cs713b` para organizar todo el proyecto.

Paso 2 - Volumen: Se creó un volumen managed llamado `noaa_raw_data` dentro del schema, con ruta: `/Volumes/labs_56754_cs713b/vessel_traffic_monitoring/noaa_raw_data/`

Paso 3 - Upload: Se subieron los 252 archivos CSV de NOAA al volumen mediante la UI de Databricks (Upload files to volume).

**Resultado Final**

Los datos NOAA están ahora almacenados en CloudLabs dentro del volumen `noaa_raw_data`. En el siguiente paso, se leerán directamente desde Spark usando la ruta del volumen, sin necesidad de acceso a S3.

**3.2. Leer archivos NOAA del volumen creado en CloudLabs**

In [0]:
# Leer archivos NOAA del volumen creado en CloudLabs

noaa_path = "/Volumes/labs_56754_cs713b/vessel_traffic_monitoring/noaa_raw_data/"

# Leer archivos CSV de NOAA
noaa_df = spark.read.csv(noaa_path, header=True, inferSchema=True)

# Mostrar esquema y primeras filas
print("Esquema NOAA:")
noaa_df.printSchema()
print("\nPrimeras filas:")
noaa_df.show(5)
print(f"\nTotal de registros NOAA: {noaa_df.count()}")

Esquema NOAA:
root
 |-- UNIQUE_ID: string (nullable = true)
 |-- FILE_UUID: string (nullable = true)
 |-- LON: double (nullable = true)
 |-- LAT: double (nullable = true)
 |-- DEPTH: double (nullable = true)
 |-- TIME: timestamp (nullable = true)
 |-- PLATFORM_NAME: string (nullable = true)
 |-- PROVIDER: string (nullable = true)


Primeras filas:
+--------------------+--------------------+----------+---------+-----+-------------------+-------------+---------+
|           UNIQUE_ID|           FILE_UUID|       LON|      LAT|DEPTH|               TIME|PLATFORM_NAME| PROVIDER|
+--------------------+--------------------+----------+---------+-----+-------------------+-------------+---------+
|ROSEP-62641c86-2f...|20250101034250876...|-90.231732|35.019627|  8.4|2024-12-31 18:53:07|  the Colonel|Rosepoint|
|ROSEP-62641c86-2f...|20250101034250876...|  -90.2317|35.019627|  8.5|2024-12-31 18:53:08|  the Colonel|Rosepoint|
|ROSEP-62641c86-2f...|20250101034250876...|-90.231668|35.019628|  8.7|2024-

**4. Crear tablas Bronze**

**4.1 Bronze - Marine Cadastre (AIS)**

In [0]:
# Guardar datos crudos de barcos en tabla Delta

ais_bronze_df = ais_df

ais_bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "labs_56754_cs713b.vessel_traffic_monitoring.bronze_ais"
)

print("✓ Tabla bronze_ais creada exitosamente")
print(f"Total registros: {ais_bronze_df.count()}")

✓ Tabla bronze_ais creada exitosamente
Total registros: 7337208


**4.2. Bronze NOAA Bathymetry**

In [0]:
# Guardar datos crudos de profundidades en tabla Delta

noaa_bronze_df = noaa_df

noaa_bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "labs_56754_cs713b.vessel_traffic_monitoring.bronze_noaa"
)

print("✓ Tabla bronze_noaa creada exitosamente")
print(f"Total registros: {noaa_bronze_df.count()}")

✓ Tabla bronze_noaa creada exitosamente
Total registros: 1133563


In [0]:
# Entender la calidad y estructura de los datos

bronze_ais = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.bronze_ais")

print("=== ESTRUCTURA ===")
bronze_ais.printSchema()

print("\n=== REGISTROS TOTALES ===")
print(f"Total: {bronze_ais.count()}")

print("\n=== NULOS POR COLUMNA ===")
for col_name in bronze_ais.columns:
    null_count = bronze_ais.filter(bronze_ais[col_name].isNull()).count()
    null_pct = (null_count / bronze_ais.count()) * 100
    print(f"{col_name}: {null_count} ({null_pct:.2f}%)")

print("\n=== RANGO DE COORDENADAS ===")
bronze_ais.select("latitude", "longitude").describe().show()

print("\n=== MUESTRA DE DATOS ===")
bronze_ais.show(3)

=== ESTRUCTURA ===
root
 |-- mmsi: integer (nullable = true)
 |-- base_date_time: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- cog: double (nullable = true)
 |-- heading: integer (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- imo: string (nullable = true)
 |-- call_sign: string (nullable = true)
 |-- vessel_type: integer (nullable = true)
 |-- status: integer (nullable = true)
 |-- length: integer (nullable = true)
 |-- width: integer (nullable = true)
 |-- draft: double (nullable = true)
 |-- cargo: integer (nullable = true)
 |-- transceiver: string (nullable = true)


=== REGISTROS TOTALES ===
Total: 7337208

=== NULOS POR COLUMNA ===
mmsi: 0 (0.00%)
base_date_time: 0 (0.00%)
longitude: 0 (0.00%)
latitude: 0 (0.00%)
sog: 15860 (0.22%)
cog: 1264519 (17.23%)
heading: 3713636 (50.61%)
vessel_name: 7520 (0.10%)
imo: 4365464 (59.50%)
call_sign: 872093 (11.89%)
vessel_

##5. Limpieza y Enriquecimiento de Datos (Silver)

**5.1. Análisis de Calidad - Marine Cadastre (AIS)**

**Datos Críticos (0% nulos)**: mmsi, base_date_time, longitude, latitude - excelente cobertura
**Datos con Nulos Significativos**: 
- heading (50.61%): Encabezamiento del barco
- imo (59.50%): Número de identificación oficial
- draft (44.06%): Calado/profundidad de navegación
- status (35.17%): Estado operacional del barco
- cargo (33.05%): Tipo de carga transportada
- cog (17.23%): Rumbo del barco

**Coordenadas**: Todas dentro de rangos válidos (lat: -90 a 90, lon: -180 a 180)

**5.2. Estrategia de Limpieza Silver - Marine Cadastre (AIS)**

In [0]:
# Explorar Bronze AIS antes de limpiar (TBD) 

**Criterio de Retención**: Mantener todos los registros que tengan mmsi, latitude, longitude y base_date_time válidos (100% de datos)

**Criterio de Tratamiento de Nulos**:
- vessel_type: Rellenar nulos con "UNKNOWN"
- cargo: Rellenar nulos con "UNKNOWN"
- Otros campos: Mantener como NULL (no son críticos para enriquecimiento)

**Razón**: El objetivo es cruzar barcos (AIS) con profundidades (NOAA) usando mmsi, ubicación y tiempo. Los campos secundarios como heading, draft y cargo son útiles para análisis pero no bloquean el ejercicio.

In [0]:
# Aplicar estrategia de limpieza definida

from pyspark.sql.functions import col, when, coalesce, lit

# Leer tabla Bronze AIS
bronze_ais = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.bronze_ais")

# Filtrar solo registros con datos críticos válidos
silver_ais = bronze_ais.filter(
    (col("mmsi").isNotNull()) &
    (col("latitude").isNotNull()) &
    (col("longitude").isNotNull()) &
    (col("base_date_time").isNotNull()) &
    (col("latitude") >= -90) & (col("latitude") <= 90) &
    (col("longitude") >= -180) & (col("longitude") <= 180)
).select(
    "*",
    when(col("vessel_type").isNull(), lit("UNKNOWN")).otherwise(col("vessel_type")).alias("vessel_type_clean"),
    when(col("cargo").isNull(), lit("UNKNOWN")).otherwise(col("cargo")).alias("cargo_clean")
).drop("vessel_type", "cargo").withColumnRenamed("vessel_type_clean", "vessel_type").withColumnRenamed("cargo_clean", "cargo")

# Guardar en Silver
silver_ais.write.format("delta").mode("overwrite").saveAsTable(
    "labs_56754_cs713b.vessel_traffic_monitoring.silver_ais"
)

print("✓ Tabla silver_ais creada")
print(f"Registros: {silver_ais.count()}")

✓ Tabla silver_ais creada
Registros: 7337208


**5.3. Análisis de Calidad - NOAA Bathymetry**

In [0]:
# Explorar Bronze NOAA antes de limpiar

bronze_noaa = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.bronze_noaa")

print("=== ESTRUCTURA ===")
bronze_noaa.printSchema()

print("\n=== REGISTROS TOTALES ===")
print(f"Total: {bronze_noaa.count()}")

print("\n=== NULOS POR COLUMNA ===")
for col_name in bronze_noaa.columns:
    null_count = bronze_noaa.filter(bronze_noaa[col_name].isNull()).count()
    null_pct = (null_count / bronze_noaa.count()) * 100
    print(f"{col_name}: {null_count} ({null_pct:.2f}%)")

print("\n=== RANGO DE COORDENADAS Y PROFUNDIDAD ===")
bronze_noaa.select("LAT", "LON", "DEPTH").describe().show()

print("\n=== MUESTRA DE DATOS ===")
bronze_noaa.show(3)

=== ESTRUCTURA ===
root
 |-- UNIQUE_ID: string (nullable = true)
 |-- FILE_UUID: string (nullable = true)
 |-- LON: double (nullable = true)
 |-- LAT: double (nullable = true)
 |-- DEPTH: double (nullable = true)
 |-- TIME: timestamp (nullable = true)
 |-- PLATFORM_NAME: string (nullable = true)
 |-- PROVIDER: string (nullable = true)


=== REGISTROS TOTALES ===
Total: 1133563

=== NULOS POR COLUMNA ===
UNIQUE_ID: 0 (0.00%)
FILE_UUID: 0 (0.00%)
LON: 0 (0.00%)
LAT: 0 (0.00%)
DEPTH: 0 (0.00%)
TIME: 0 (0.00%)
PLATFORM_NAME: 0 (0.00%)
PROVIDER: 0 (0.00%)

=== RANGO DE COORDENADAS Y PROFUNDIDAD ===
+-------+------------------+------------------+------------------+
|summary|               LAT|               LON|             DEPTH|
+-------+------------------+------------------+------------------+
|  count|           1133563|           1133563|           1133563|
|   mean| 33.88284813562753|-83.11433088825275|  8.01879519351502|
| stddev| 7.675094212361369|17.884479736630112|12.25137146797450

**Datos**: 100% completos, 0% nulos en todas las columnas
- UNIQUE_ID, FILE_UUID, LON, LAT, DEPTH, TIME, PLATFORM_NAME, PROVIDER: sin nulos

**Coordenadas y Profundidad**:
- Latitud: 18° a 61° (Golfo de México a Canadá)
- Longitud: -123° a 5° (Pacífico a Atlántico)
- Profundidad: 0.3m a 157.1m (aguas someras a moderadamente profundas)

**Estrategia Silver**: 
- Mantener todos los registros (datos limpios)
- Validar rangos de coordenadas
- Convertir columnas a minúsculas para consistencia con AIS

In [0]:
# Limpiar NOAA
# Validar y estandarizar formato

from pyspark.sql.functions import col

# Leer tabla Bronze NOAA
bronze_noaa = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.bronze_noaa")

# Validar rangos y renombrar columnas a minúsculas
silver_noaa = bronze_noaa.filter(
    (col("LAT") >= -90) & (col("LAT") <= 90) &
    (col("LON") >= -180) & (col("LON") <= 180) &
    (col("DEPTH") > 0)
).select(
    col("UNIQUE_ID").alias("unique_id"),
    col("FILE_UUID").alias("file_uuid"),
    col("LON").alias("lon"),
    col("LAT").alias("lat"),
    col("DEPTH").alias("depth"),
    col("TIME").alias("time"),
    col("PLATFORM_NAME").alias("platform_name"),
    col("PROVIDER").alias("provider")
)

# Guardar en Silver
silver_noaa.write.format("delta").mode("overwrite").saveAsTable(
    "labs_56754_cs713b.vessel_traffic_monitoring.silver_noaa"
)

print("✓ Tabla silver_noaa creada")
print(f"Registros: {silver_noaa.count()}")

✓ Tabla silver_noaa creada
Registros: 1133563


**5.3 Enriquecimiento - Cruce AIS + NOAA**

**H3: Indexación Geográfica Hexagonal**

H3 divide el mundo en celdas hexagonales de tamaño uniforme (~1200 km² en nivel 5).
Convierte cada coordenada lat/lon a su celda H3 correspondiente.

**Es crítico porque**: Sin H3, cruzar AIS con NOAA sería imposible.
Las coordenadas de barcos y profundidades nunca coinciden exactamente.
Con H3, agrupamos ambos datasets en las MISMAS celdas hexagonales,
permitiendo un JOIN rápido: "barcos en celda X + profundidades en celda X".

**Estrategia**:
1. Agregar columnas H3 (grilla hexagonal de nivel 5) a ambas tablas
2. Crear ventanas de tiempo (30 minutos) para agrupar mediciones cercanas
3. Cruzar AIS con NOAA: mismo H3 + mismo período de tiempo
4. Calcular profundidad promedio por barco/zona/tiempo
5. Enriquecer cada posición AIS con profundidad cercana

**Resultado**: Tabla que dice "El barco X en ubicación Y en tiempo Z, navegaba en aguas de profundidad promedio W metros"

In [0]:
# Instalar H3

%pip install h3

from h3 import latlng_to_cell

print("✓ H3 instalado correctamente")

# Prueba rápida
test_h3 = latlng_to_cell(35.0, -90.0, 5)
print(f"Ejemplo H3: lat=35, lon=-90, nivel=5 → {test_h3}")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✓ H3 instalado correctamente
Ejemplo H3: lat=35, lon=-90, nivel=5 → 852659affffffff


In [0]:
# Agregar indexación H3 a Silver AIS y NOAA

from h3 import latlng_to_cell
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# Crear función UDF para H3
h3_udf = udf(lambda lat, lon: latlng_to_cell(lat, lon, 5), StringType())

# Leer tablas Silver
silver_ais = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.silver_ais")
silver_noaa = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.silver_noaa")

# Agregar H3 a AIS
silver_ais_h3 = silver_ais.withColumn("h3_cell", h3_udf(col("latitude"), col("longitude")))

# Agregar H3 a NOAA
silver_noaa_h3 = silver_noaa.withColumn("h3_cell", h3_udf(col("lat"), col("lon")))

print("✓ H3 agregado a AIS")
print(f"Muestra AIS con H3:")
silver_ais_h3.select("mmsi", "latitude", "longitude", "h3_cell").show(3)

print("\n✓ H3 agregado a NOAA")
print(f"Muestra NOAA con H3:")
silver_noaa_h3.select("unique_id", "lat", "lon", "depth", "h3_cell").show(3)

✓ H3 agregado a AIS
Muestra AIS con H3:
+---------+--------+----------+---------------+
|     mmsi|latitude| longitude|        h3_cell|
+---------+--------+----------+---------------+
|671087100|18.46281| -66.10297|854cee4ffffffff|
|367733950|48.48503|-122.60927|8528d11bfffffff|
|368138010|40.47715| -73.84652|852a102bfffffff|
+---------+--------+----------+---------------+
only showing top 3 rows


✓ H3 agregado a NOAA
Muestra NOAA con H3:
+--------------------+------------------+------------------+-----+---------------+
|           unique_id|               lat|               lon|depth|        h3_cell|
+--------------------+------------------+------------------+-----+---------------+
|AQM-65f387283db57...|18.305286000000002|-65.29925866666667|  4.5|854ce857fffffff|
|AQM-65f387283db57...|18.305286333333335|       -65.2992595|  4.5|854ce857fffffff|
|AQM-65f387283db57...|18.305286833333334|         -65.29926|  4.5|854ce857fffffff|
+--------------------+------------------+-----------------

In [0]:
# Cruzar AIS con NOAA - Crear tabla Silver Enriquecida

from pyspark.sql.functions import col, avg, count, window, round as spark_round

# Datos con H3
silver_ais_h3 = silver_ais_h3
silver_noaa_h3 = silver_noaa_h3

# Paso 1: Agregar NOAA por H3 + ventana de 30 minutos
noaa_aggregated = silver_noaa_h3.groupBy(
    col("h3_cell"),
    window(col("time"), "30 minutes").alias("time_window")
).agg(
    avg(col("depth")).alias("avg_depth"),
    spark_round(avg(col("depth")), 2).alias("avg_depth_rounded"),
    count(col("depth")).alias("depth_measurements")
).select(
    col("h3_cell").alias("noaa_h3_cell"),
    col("time_window.start").alias("time_start"),
    col("time_window.end").alias("time_end"),
    col("avg_depth_rounded"),
    col("depth_measurements")
)

print("✓ NOAA agregado por H3 + tiempo")
print(f"Registros agregados NOAA: {noaa_aggregated.count()}")
noaa_aggregated.show(3)

# Paso 2: Crear ventana de tiempo en AIS
ais_with_window = silver_ais_h3.withColumn(
    "time_window",
    window(col("base_date_time"), "30 minutes")
)

# Paso 3: Cruzar AIS con NOAA agregado
silver_enriched = ais_with_window.join(
    noaa_aggregated,
    (ais_with_window.h3_cell == noaa_aggregated.noaa_h3_cell) &
    (ais_with_window.time_window.start <= noaa_aggregated.time_end) &
    (ais_with_window.time_window.end >= noaa_aggregated.time_start),
    "left"
).select(
    ais_with_window.mmsi,
    ais_with_window.vessel_name,
    ais_with_window.vessel_type,
    ais_with_window.cargo,
    ais_with_window.base_date_time,
    ais_with_window.latitude,
    ais_with_window.longitude,
    ais_with_window.h3_cell,
    noaa_aggregated.avg_depth_rounded.alias("nearby_depth_m"),
    noaa_aggregated.depth_measurements.alias("depth_samples")
)

print("\n✓ AIS enriquecido con profundidades NOAA")
print(f"Registros enriquecidos: {silver_enriched.count()}")
silver_enriched.show(5)

# Paso 4: Guardar tabla Silver enriquecida
silver_enriched.write.format("delta").mode("overwrite").saveAsTable(
    "labs_56754_cs713b.vessel_traffic_monitoring.silver_enriched"
)

print("\n✓ Tabla silver_enriched guardada correctamente")

✓ NOAA agregado por H3 + tiempo
Registros agregados NOAA: 1370
+---------------+-------------------+-------------------+-----------------+------------------+
|   noaa_h3_cell|         time_start|           time_end|avg_depth_rounded|depth_measurements|
+---------------+-------------------+-------------------+-----------------+------------------+
|8509abcbfffffff|2025-01-01 00:00:00|2025-01-01 00:30:00|              8.9|               600|
|85444443fffffff|2025-01-01 04:30:00|2025-01-01 05:00:00|             3.33|              1737|
|85444003fffffff|2025-01-01 06:30:00|2025-01-01 07:00:00|            11.71|              1493|
+---------------+-------------------+-------------------+-----------------+------------------+
only showing top 3 rows


✓ AIS enriquecido con profundidades NOAA
Registros enriquecidos: 8292498
+---------+--------------+-----------+-------+-------------------+--------+----------+---------------+--------------+-------------+
|     mmsi|   vessel_name|vessel_type|  c

## 6. Agregación y Métricas Finales (Gold) 

In [0]:
# Agregación y Métricas Finales

from pyspark.sql.functions import col, avg, count, min, max, round as spark_round

# Leer tabla Silver enriquecida
silver_enriched = spark.table("labs_56754_cs713b.vessel_traffic_monitoring.silver_enriched")

# Agregación por H3 cell (zona geográfica) + tipo barco + carga
gold = silver_enriched.groupBy(
    col("h3_cell"),
    col("latitude"),
    col("longitude"),
    col("vessel_type"),
    col("cargo")
).agg(
    count(col("mmsi")).alias("vessel_count"),
    avg(col("nearby_depth_m")).alias("avg_depth_m"),
    min(col("nearby_depth_m")).alias("min_depth_m"),
    max(col("nearby_depth_m")).alias("max_depth_m"),
    avg(col("depth_samples")).alias("avg_samples")
).select(
    col("h3_cell"),
    spark_round(col("latitude"), 4).alias("latitude"),
    spark_round(col("longitude"), 4).alias("longitude"),
    col("vessel_type"),
    col("cargo"),
    col("vessel_count"),
    spark_round(col("avg_depth_m"), 2).alias("avg_depth_m"),
    spark_round(col("min_depth_m"), 2).alias("min_depth_m"),
    spark_round(col("max_depth_m"), 2).alias("max_depth_m"),
    spark_round(col("avg_samples"), 2).alias("avg_samples")
)

print("✓ Tabla Gold creada")
print(f"Registros Gold: {gold.count()}")
gold.show(10)

# Guardar Gold
gold.write.format("delta").mode("overwrite").saveAsTable(
    "labs_56754_cs713b.vessel_traffic_monitoring.gold_analytics"
)

print("\n✓ Tabla gold_analytics guardada correctamente")

✓ Tabla Gold creada
Registros Gold: 3266827
+---------------+--------+---------+-----------+-------+------------+-----------+-----------+-----------+-----------+
|        h3_cell|latitude|longitude|vessel_type|  cargo|vessel_count|avg_depth_m|min_depth_m|max_depth_m|avg_samples|
+---------------+--------+---------+-----------+-------+------------+-----------+-----------+-----------+-----------+
|8528d557fffffff| 47.6282|-122.3941|         36|UNKNOWN|           2|       NULL|       NULL|       NULL|       NULL|
|852aac77fffffff| 39.5728| -76.1028|         52|     52|           4|       NULL|       NULL|       NULL|       NULL|
|85444643fffffff| 29.9973| -90.0241|         31|     31|           6|       4.23|       3.36|       7.52|      45.33|
|852aad3bfffffff| 38.7828| -75.1203|         50|     50|           8|       NULL|       NULL|       NULL|       NULL|
|85444603fffffff| 29.9928| -90.4376|         52|     52|           1|       NULL|       NULL|       NULL|       NULL|
|8528d543fff

**Consultas para Dashboard**

Las siguientes consultas se definen en el Dashboard (pestaña "Data"):

- Query 1: Summary Data Set
- Query 2: Mapa H3 con profundidades
- Query 3: Barcos por tipo
- Query 4: Barcos por carga  
- Query 5: Zonas de riesgo

Estas consultas consultan la tabla `gold_analytics` creada en esta fase.
NO se ejecutan en el notebook, solo en el dashboard.